# Tedarik Zinciri SQL Analizi


**Veri:** [Kaggle — Supply Chain Data](https://www.kaggle.com/datasets/laurinbrechter/supply-chain-data)  
**Araçlar:** SQL (SQLite), Python, Pandas, Matplotlib  

---

## İçerik
1. Veritabanı Kurulumu
2. Temel SQL Sorguları
3. İleri Seviye SQL (JOIN, GROUP BY, WINDOW FUNCTIONS)
4. Stok & Tedarikçi Analizi
5. Görselleştirme


## 1️.Veritabanı Kurulumu

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings_imported = True

# Colab'da dosya yükleme
# from google.colab import files
# uploaded = files.upload()

# CSV'yi oku ve SQLite veritabanına yükle
df = pd.read_csv('supply_chain_data.csv')

# SQLite bağlantısı kur (bellekte)
conn = sqlite3.connect(':memory:')

# Tablo adları için sütun isimlerini düzenle
df.columns = [c.lower().replace(' ', '_').replace('[', '').replace(']', '')
              .replace('/', '_').replace('-', '_') for c in df.columns]

# Veriyi veritabanına yükle
df.to_sql('supply_chain', conn, if_exists='replace', index=False)

print("✓ Veritabanı kuruldu!")
print(f"  Tablo: supply_chain")
print(f"  Kayıt sayısı: {len(df)}")
print(f"  Sütunlar: {list(df.columns)}")


In [ ]:
# Yardımcı fonksiyon: SQL sorgusunu çalıştır ve sonucu göster
def sql(query, show=True):
    result = pd.read_sql_query(query, conn)
    if show:
        display(result)
    return result

# Tablo yapısını incele
sql("PRAGMA table_info(supply_chain)")


## 2️.Temel SQL Sorguları

### Sorgu 1 — Genel Bakış


In [ ]:
-- Genel istatistikler
sql("""
SELECT 
    COUNT(*)                        AS toplam_urun,
    COUNT(DISTINCT supplier_name)   AS tedarikci_sayisi,
    COUNT(DISTINCT location)        AS lokasyon_sayisi,
    COUNT(DISTINCT product_type)    AS urun_tipi_sayisi,
    ROUND(AVG(price), 2)            AS ort_fiyat,
    ROUND(SUM(revenue_generated), 0) AS toplam_gelir,
    ROUND(AVG(defect_rates), 3)     AS ort_hata_orani,
    ROUND(AVG(stock_levels), 1)     AS ort_stok_seviyesi
FROM supply_chain
""")


### Sorgu 2 — Ürün Tipi Bazlı Özet

In [ ]:
sql("""
SELECT 
    product_type                            AS urun_tipi,
    COUNT(*)                                AS urun_sayisi,
    ROUND(AVG(price), 2)                    AS ort_fiyat,
    ROUND(SUM(revenue_generated), 0)        AS toplam_gelir,
    ROUND(AVG(stock_levels), 1)             AS ort_stok,
    ROUND(AVG(defect_rates), 3)             AS ort_hata_orani,
    ROUND(AVG(lead_times), 1)               AS ort_temin_suresi
FROM supply_chain
GROUP BY product_type
ORDER BY toplam_gelir DESC
""")


### Sorgu 3 — Tedarikçi Performans Skorecard

In [ ]:
sql("""
SELECT 
    supplier_name                               AS tedarikci,
    COUNT(*)                                    AS urun_sayisi,
    ROUND(AVG(lead_time), 1)                    AS ort_temin_suresi,
    ROUND(AVG(defect_rates), 3)                 AS ort_hata_orani,
    ROUND(AVG(production_volumes), 0)           AS ort_uretim_hacmi,
    ROUND(SUM(revenue_generated), 0)            AS toplam_gelir,
    ROUND(AVG(manufacturing_costs), 2)          AS ort_uretim_maliyeti,
    ROUND(
        (30 - AVG(lead_time)) / 30.0 * 40 +
        (5  - AVG(defect_rates)) / 5.0 * 40 +
        AVG(production_volumes) / 985.0 * 20
    , 1)                                        AS performans_skoru
FROM supply_chain
GROUP BY supplier_name
ORDER BY performans_skoru DESC
""")


### Sorgu 4 — Stok Durumu & Yeniden Sipariş Analizi

In [ ]:
sql("""
SELECT 
    sku,
    product_type,
    supplier_name,
    stock_levels                                AS mevcut_stok,
    lead_times                                  AS temin_suresi_gun,
    order_quantities                            AS siparis_miktari,
    ROUND(number_of_products_sold / 52.0, 1)   AS haftalik_talep,
    ROUND(number_of_products_sold / 52.0 
          * lead_times / 7.0, 0)               AS yeniden_siparis_noktasi,
    CASE 
        WHEN stock_levels < (number_of_products_sold / 52.0 * lead_times / 7.0)
        THEN 'KRITIK - Siparis Ver!'
        WHEN stock_levels < (number_of_products_sold / 52.0 * lead_times / 7.0) * 1.5
        THEN 'Dusuk - Izle'
        ELSE 'Normal'
    END                                         AS stok_durumu
FROM supply_chain
ORDER BY stok_durumu DESC, stock_levels ASC
LIMIT 15
""")


## 3️.İleri Seviye SQL

### Sorgu 5 — ABC Analizi (Kümülatif Gelir)


In [ ]:
sql("""
WITH gelir_sirali AS (
    SELECT 
        sku,
        product_type,
        supplier_name,
        revenue_generated,
        SUM(revenue_generated) OVER (ORDER BY revenue_generated DESC) AS kumulatif_gelir,
        SUM(revenue_generated) OVER ()                                 AS toplam_gelir
    FROM supply_chain
),
abc AS (
    SELECT *,
        ROUND(kumulatif_gelir / toplam_gelir * 100, 2) AS kumulatif_pct,
        CASE 
            WHEN kumulatif_gelir / toplam_gelir * 100 <= 80 THEN 'A - Yuksek Deger'
            WHEN kumulatif_gelir / toplam_gelir * 100 <= 95 THEN 'B - Orta Deger'
            ELSE 'C - Dusuk Deger'
        END AS abc_sinifi
    FROM gelir_sirali
)
SELECT 
    abc_sinifi,
    COUNT(*)                            AS urun_sayisi,
    ROUND(SUM(revenue_generated), 0)    AS toplam_gelir,
    ROUND(AVG(revenue_generated), 0)    AS ort_gelir
FROM abc
GROUP BY abc_sinifi
ORDER BY toplam_gelir DESC
""")


### Sorgu 6 — Taşıma Modu Maliyet & Verimlilik Analizi

In [ ]:
sql("""
SELECT 
    transportation_modes                        AS tasima_modu,
    COUNT(*)                                    AS siparis_sayisi,
    ROUND(AVG(shipping_costs), 2)               AS ort_kargo_maliyeti,
    ROUND(AVG(shipping_times), 1)               AS ort_teslimat_suresi,
    ROUND(AVG(revenue_generated), 0)            AS ort_gelir,
    ROUND(AVG(shipping_costs) / 
          AVG(revenue_generated) * 100, 2)      AS kargo_gelir_orani_pct,
    ROUND(AVG(defect_rates), 3)                 AS ort_hata_orani
FROM supply_chain
GROUP BY transportation_modes
ORDER BY ort_kargo_maliyeti ASC
""")


### Sorgu 7 — WINDOW FUNCTION: Tedarikçi İçinde Ürün Sıralaması

In [ ]:
sql("""
SELECT 
    supplier_name,
    sku,
    product_type,
    revenue_generated,
    RANK() OVER (
        PARTITION BY supplier_name 
        ORDER BY revenue_generated DESC
    )                                           AS tedarikci_ici_sira,
    ROUND(revenue_generated / SUM(revenue_generated) 
          OVER (PARTITION BY supplier_name) * 100, 1) AS tedarikci_gelir_payi_pct
FROM supply_chain
ORDER BY supplier_name, tedarikci_ici_sira
LIMIT 20
""")


### Sorgu 8 — Kritik Tedarikçi Riski (Tek Tedarikçiye Bağımlılık)

In [ ]:
sql("""
WITH urun_tedarikci AS (
    SELECT 
        product_type,
        supplier_name,
        COUNT(*) AS urun_sayisi,
        ROUND(SUM(revenue_generated), 0) AS gelir
    FROM supply_chain
    GROUP BY product_type, supplier_name
),
toplam AS (
    SELECT product_type, SUM(urun_sayisi) AS toplam
    FROM urun_tedarikci
    GROUP BY product_type
)
SELECT 
    u.product_type                  AS urun_tipi,
    u.supplier_name                 AS tedarikci,
    u.urun_sayisi,
    u.gelir,
    ROUND(u.urun_sayisi * 100.0 
          / t.toplam, 1)            AS pazar_payi_pct,
    CASE 
        WHEN u.urun_sayisi * 100.0 / t.toplam >= 60 
        THEN 'YUKSEK RiSK - Tek Tedarikci'
        WHEN u.urun_sayisi * 100.0 / t.toplam >= 40 
        THEN 'ORTA RiSK'
        ELSE 'NORMAL'
    END                             AS risk_seviyesi
FROM urun_tedarikci u
JOIN toplam t ON u.product_type = t.product_type
ORDER BY pazar_payi_pct DESC
LIMIT 15
""")


## 4️.SQL Sonuçlarının Görselleştirilmesi

In [ ]:
# Tüm analizleri görselleştir
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Tedarik Zinciri SQL Analizi — Görsel Özet', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
colors = ['#2E75B6','#E24B4A','#1F9E75','#F2A623','#7030A0']

# 1. Tedarikçi performans skoru
ax1 = fig.add_subplot(gs[0, 0])
perf = sql("""
    SELECT supplier_name,
           ROUND((30-AVG(lead_time))/30.0*40 +
                 (5-AVG(defect_rates))/5.0*40 +
                 AVG(production_volumes)/985.0*20, 1) AS skor
    FROM supply_chain GROUP BY supplier_name ORDER BY skor DESC
""", show=False)
bars = ax1.bar(perf['supplier_name'], perf['skor'],
               color=colors[:len(perf)], alpha=0.85)
ax1.set_title('Tedarikçi Performans Skoru', fontweight='bold')
ax1.set_ylabel('Skor')
ax1.set_ylim(0, 100)
ax1.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, perf['skor']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val}', ha='center', fontsize=9, fontweight='bold')

# 2. Ürün tipi gelir dağılımı
ax2 = fig.add_subplot(gs[0, 1])
gelir = sql("""
    SELECT product_type, ROUND(SUM(revenue_generated),0) AS toplam_gelir
    FROM supply_chain GROUP BY product_type ORDER BY toplam_gelir DESC
""", show=False)
ax2.pie(gelir['toplam_gelir'], labels=gelir['product_type'],
        autopct='%1.1f%%', colors=colors[:len(gelir)], startangle=90)
ax2.set_title('Ürün Tipi Gelir Dağılımı', fontweight='bold')

# 3. Stok durumu
ax3 = fig.add_subplot(gs[0, 2])
stok = sql("""
    SELECT 
        CASE 
            WHEN stock_levels < (number_of_products_sold/52.0*lead_times/7.0)
            THEN 'Kritik'
            WHEN stock_levels < (number_of_products_sold/52.0*lead_times/7.0)*1.5
            THEN 'Dusuk'
            ELSE 'Normal'
        END AS durum, COUNT(*) AS sayi
    FROM supply_chain GROUP BY durum
""", show=False)
stok_colors = {'Kritik':'#E24B4A','Dusuk':'#F2A623','Normal':'#1F9E75'}
bar_colors  = [stok_colors.get(d,'#2E75B6') for d in stok['durum']]
bars3 = ax3.bar(stok['durum'], stok['sayi'], color=bar_colors, alpha=0.85)
ax3.set_title('Stok Durumu Dağılımı', fontweight='bold')
ax3.set_ylabel('Ürün Sayısı')
for bar, val in zip(bars3, stok['sayi']):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             str(val), ha='center', fontsize=12, fontweight='bold')

# 4. ABC analizi
ax4 = fig.add_subplot(gs[1, 0])
abc = sql("""
    WITH gs AS (
        SELECT sku, revenue_generated,
               SUM(revenue_generated) OVER (ORDER BY revenue_generated DESC) AS cum,
               SUM(revenue_generated) OVER () AS total
        FROM supply_chain
    )
    SELECT CASE WHEN cum/total*100<=80 THEN 'A'
                WHEN cum/total*100<=95 THEN 'B'
                ELSE 'C' END AS sinif, COUNT(*) AS sayi
    FROM gs GROUP BY sinif ORDER BY sinif
""", show=False)
ax4.pie(abc['sayi'], labels=[f'Sınıf {s}' for s in abc['sinif']],
        autopct='%1.1f%%', colors=['#1F9E75','#F2A623','#E24B4A'],
        startangle=90, explode=[0.05,0,0])
ax4.set_title('ABC Analizi (SKU Sayısı)', fontweight='bold')

# 5. Taşıma modu karşılaştırması
ax5 = fig.add_subplot(gs[1, 1])
trans = sql("""
    SELECT transportation_modes AS mod,
           ROUND(AVG(shipping_costs),2) AS maliyet,
           ROUND(AVG(shipping_times),1) AS sure
    FROM supply_chain GROUP BY transportation_modes
""", show=False)
x = np.arange(len(trans))
w = 0.35
ax5.bar(x-w/2, trans['maliyet'], w, label='Kargo Maliyeti', color='#E24B4A', alpha=0.85)
ax5.bar(x+w/2, trans['sure']*10, w, label='Süre×10', color='#2E75B6', alpha=0.85)
ax5.set_xticks(x)
ax5.set_xticklabels(trans['mod'], rotation=15, fontsize=9)
ax5.set_title('Taşıma Modu Analizi', fontweight='bold')
ax5.legend(fontsize=8)

# 6. Hata oranı vs temin süresi
ax6 = fig.add_subplot(gs[1, 2])
scatter_data = sql("""
    SELECT supplier_name, AVG(lead_times) AS lt,
           AVG(defect_rates) AS hata,
           SUM(revenue_generated) AS gelir
    FROM supply_chain GROUP BY supplier_name
""", show=False)
for i, row in scatter_data.iterrows():
    ax6.scatter(row['lt'], row['hata'], s=row['gelir']/500,
                color=colors[i], alpha=0.8, zorder=5)
    ax6.annotate(row['supplier_name'].replace('Supplier ','S'),
                 (row['lt'], row['hata']), fontsize=8, ha='center', va='bottom')
ax6.set_xlabel('Ort. Temin Süresi (gün)')
ax6.set_ylabel('Ort. Hata Oranı')
ax6.set_title('Temin Süresi vs Hata Oranı
(Balon = Gelir)', fontweight='bold')

plt.tight_layout()
plt.savefig('sql_analiz_grafik.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Görselleştirme tamamlandı!")


## 5️.Özet & SQL Sorgu Listesi

In [ ]:
print("=" * 55)
print("TEDARİK ZİNCİRİ SQL ANALİZİ — ÖZET")
print("=" * 55)

ozet = sql("""
    SELECT
        COUNT(*)                                AS toplam_sku,
        ROUND(SUM(revenue_generated), 0)        AS toplam_gelir,
        ROUND(AVG(defect_rates)*100, 2)         AS ort_hata_orani_pct,
        ROUND(AVG(lead_times), 1)               AS ort_temin_suresi,
        SUM(CASE WHEN stock_levels < 
            (number_of_products_sold/52.0*lead_times/7.0)
            THEN 1 ELSE 0 END)                  AS kritik_stok_sayisi
    FROM supply_chain
""", show=False)

print(f"  Toplam SKU        : {ozet['toplam_sku'][0]}")
print(f"  Toplam Gelir      : {ozet['toplam_gelir'][0]:,.0f}")
print(f"  Ort. Hata Oranı   : %{ozet['ort_hata_orani_pct'][0]}")
print(f"  Ort. Temin Süresi : {ozet['ort_temin_suresi'][0]} gün")
print(f"  Kritik Stok       : {ozet['kritik_stok_sayisi'][0]} ürün")

print("\nKULLANILAN SQL TEKNİKLERİ:")
print("  - SELECT, WHERE, GROUP BY, ORDER BY, LIMIT")
print("  - CASE WHEN / THEN / ELSE")
print("  - Aggregate functions: AVG, SUM, COUNT, ROUND")
print("  - CTE (WITH ... AS)")
print("  - WINDOW FUNCTIONS: RANK(), SUM() OVER(PARTITION BY)")
print("  - JOIN (implicit via CTE)")
print("  - Subqueries")
print("\n✓ Tüm analizler tamamlandı!")
